## MCP reserved method names (JSON-RPC)

On the wire, MCP is JSON-RPC. The strings below are **reserved method names** from the spec — not GitHub-specific. Python `session.list_tools()` is only a wrapper around `"tools/list"`.

A real host (Claude, Codex) speaks these names. GitHub MCP answers the ones its handshake **capabilities** advertised.

### Lifecycle (every connection)

| Method | Direction | What it is |
|--------|-----------|------------|
| `initialize` | client → server | First request: protocol version, clientInfo, client capabilities |
| `notifications/initialized` | client → server | Handshake done; normal requests may follow |
| `ping` | either way | Liveness check |
| `notifications/cancelled` | either way | Cancel an in-flight request |
| `notifications/progress` | either way | Progress for a long call |

### Tools

| Method | What it is |
|--------|------------|
| `tools/list` | Catalog of tools (name, description, JSON Schema) |
| `tools/call` | Run one tool |
| `notifications/tools/list_changed` | Server says the catalog changed; host should list again |

### Resources

| Method | What it is |
|--------|------------|
| `resources/list` | Concrete URIs you can read now |
| `resources/templates/list` | URI patterns with `{placeholders}` |
| `resources/read` | Fetch one URI (file text, HTML UI, …) |
| `resources/subscribe` / `unsubscribe` | Watch a URI for updates |
| `notifications/resources/list_changed` | Catalog changed |
| `notifications/resources/updated` | A subscribed URI changed |

### Prompts

| Method | What it is |
|--------|------------|
| `prompts/list` | Named recipes + argument schema |
| `prompts/get` | Expand a recipe into messages |
| `notifications/prompts/list_changed` | Catalog changed |

### Other (optional capabilities)

| Method | What it is |
|--------|------------|
| `completion/complete` | Autocomplete a prompt arg or URI template placeholder |
| `logging/setLevel` | Ask the server to emit `notifications/message` at a log level |
| `notifications/message` | Server log line (not the handshake) |

Server → client (this notebook does **not** implement these): `sampling/createMessage`, `roots/list`, `elicitation/create`.

The next cell prints the JSON-RPC **method string** on every request, then explores `ping`, `tools/list`, and `completion/complete`.


In [17]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()

REMOTE_MCP_URL = "https://api.githubcopilot.com/mcp/"
TOOLSETS = "repos,issues,pull_requests,context,users,git"
SERVER_NAME = "github"
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


In [4]:
def build_github_connection() -> dict:
    headers = {
        "X-MCP-Toolsets": TOOLSETS,
        "X-MCP-Readonly": "true",
        "Authorization": f"Bearer {GITHUB_TOKEN}"
    }
    return {
        SERVER_NAME: {
            "url": REMOTE_MCP_URL,
            "headers": headers,
            "transport": "http",
            "timeout": 60,
            "sse_read_timeout": 300
        }
    }

In [5]:
CONNECTIONS = build_github_connection()

In [7]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [8]:
mcp_client = MultiServerMCPClient(CONNECTIONS)

In [9]:
# Inspect connections without printing the live token.
{
    name: {
        **cfg,
        "headers": {
            key: ("Bearer <GITHUB_TOKEN>" if key.lower() == "authorization" else value)
            for key, value in (cfg.get("headers") or {}).items()
        },
    }
    for name, cfg in mcp_client.connections.items()
}

{'github': {'url': 'https://api.githubcopilot.com/mcp/',
  'headers': {'X-MCP-Toolsets': 'repos,issues,pull_requests,context,users,git',
   'X-MCP-Readonly': 'true',
   'Authorization': 'Bearer <GITHUB_TOKEN>'},
  'transport': 'http',
  'timeout': 60,
  'sse_read_timeout': 300}}

In [ ]:
def parse_tools(tools_result):
    rows = []

    for tool in tools_result.tools:
        schema = getattr(tool, "inputSchema", None) or {}
        if hasattr(schema, "model_dump"):
            schema = schema.model_dump()
        rows.append({
            "name": tool.name,
            "description": tool.description,
            "input_schema": schema if isinstance(schema, dict) else {},
        })
    
    return sorted(rows, key=lambda x: x["name"])

    

In [100]:
async def fetch_catalog(client, server_name: str) -> dict:
    async with client.session(server_name) as session:
        # tools = await session.get_tools()
        resources = await session.list_resources()
        print(resources.resources)
        prompts = await session.list_prompts()
        print(prompts.prompts[-1])
        print(len(prompts.prompts))
    
    return None

In [101]:
await fetch_catalog(mcp_client, SERVER_NAME)

[Resource(name='get_me_ui', title=None, uri=AnyUrl('ui://github-mcp-server/get-me'), description='MCP App UI for the get_me tool', mimeType='text/html;profile=mcp-app', size=None, icons=None, annotations=None, meta=None)]
name='issue_to_fix_workflow' title=None description='Create an issue for a problem and then generate a pull request to fix it' arguments=[PromptArgument(name='owner', description='Repository owner', required=True), PromptArgument(name='repo', description='Repository name', required=True), PromptArgument(name='title', description='Issue title', required=True), PromptArgument(name='description', description='Issue description', required=True), PromptArgument(name='labels', description='Comma-separated list of labels to apply (optional)', required=None), PromptArgument(name='assignees', description='Comma-separated list of assignees (optional)', required=None)] icons=[Icon(src='', mimeType='image/png', sizes=None, theme='light'), Icon(src='', mimeType='image/png', sizes=

In [77]:
response.tools

[Tool(name='get_commit', title=None, description='Get details for a commit from a GitHub repository', inputSchema={'type': 'object', 'properties': {'detail': {'type': 'string', 'description': 'Level of detail to include for changed files. "none" omits stats and files entirely. "stats" (default) includes per-file metadata: filename, status, and lines-of-code counts (additions, deletions, changes), with no patch content. "full_patch" additionally includes the unified diff content for each file and can be very large.', 'default': 'stats', 'enum': ['none', 'stats', 'full_patch']}, 'owner': {'type': 'string', 'description': 'Repository owner', 'x-mcp-header': 'owner'}, 'page': {'type': 'number', 'description': 'Page number for pagination (min 1)', 'minimum': 1}, 'perPage': {'type': 'number', 'description': 'Results per page for pagination (min 1, max 100)', 'minimum': 1, 'maximum': 100}, 'repo': {'type': 'string', 'description': 'Repository name', 'x-mcp-header': 'repo'}, 'sha': {'type': 's

In [20]:
tools_df = pd.DataFrame(
    [
        {
            "name": t["name"],
            "description": t["description"],
            "input_schema": t["input_schema"],
        } for t in response["tools"]
    ]
)
tools_df 

,name,description,input_schema
0,get_commit,Get details for a commit from a GitHub repository,"{'type': 'object', 'properties': {'detail': {'..."
1,get_file_contents,Get the contents of a file or directory from a...,"{'type': 'object', 'properties': {'fields': {'..."
2,get_label,Get a specific label from a repository.,"{'type': 'object', 'properties': {'name': {'ty..."
3,get_latest_release,Get the latest release in a GitHub repository,"{'type': 'object', 'properties': {'owner': {'t..."
4,get_me,Get details of the authenticated GitHub user. ...,"{'type': 'object', 'properties': {}}"
5,get_release_by_tag,Get a specific release by its tag name in a Gi...,"{'type': 'object', 'properties': {'owner': {'t..."
6,get_repository_tree,Get the tree structure (files and directories)...,"{'type': 'object', 'properties': {'owner': {'t..."
7,get_tag,Get details about a specific git tag in a GitH...,"{'type': 'object', 'properties': {'owner': {'t..."
8,get_team_members,Get member usernames of a specific team in an ...,"{'type': 'object', 'properties': {'org': {'typ..."
9,get_teams,Get details of the teams the user is a member ...,"{'type': 'object', 'properties': {'user': {'ty..."


In [63]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

In [64]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [80]:
tools = await mcp_client.get_tools(server_name=SERVER_NAME)

tools

[StructuredTool(name='get_commit', description='Get details for a commit from a GitHub repository', args_schema={'type': 'object', 'properties': {'detail': {'type': 'string', 'description': 'Level of detail to include for changed files. "none" omits stats and files entirely. "stats" (default) includes per-file metadata: filename, status, and lines-of-code counts (additions, deletions, changes), with no patch content. "full_patch" additionally includes the unified diff content for each file and can be very large.', 'default': 'stats', 'enum': ['none', 'stats', 'full_patch']}, 'owner': {'type': 'string', 'description': 'Repository owner', 'x-mcp-header': 'owner'}, 'page': {'type': 'number', 'description': 'Page number for pagination (min 1)', 'minimum': 1}, 'perPage': {'type': 'number', 'description': 'Results per page for pagination (min 1, max 100)', 'minimum': 1, 'maximum': 100}, 'repo': {'type': 'string', 'description': 'Repository name', 'x-mcp-header': 'repo'}, 'sha': {'type': 'str

In [81]:
system_prompt = """
You are a github assistant. Use mcp tools for live data. Do not create any PRs or repos. 
"""

agent = create_agent(
    model, 
    tools=tools,
    system_prompt=system_prompt,
)

In [82]:
question="Who am I on github? Name a few repositories if available"

output = await agent.ainvoke({
    "messages": [
        {"role": "user", "content": question},
    ],
})


In [83]:
print(output["messages"][-1].content)

You are [Sanket Singh](https://github.com/singhsanket143) on GitHub. Here are a few of your repositories:

1. **[CppCompetitiveRepository](https://github.com/singhsanket143/CppCompetitiveRepository)**  
   Description: Repository for codes of various data structures and algorithms and competitive programming problems.  
   Language: C++  
   Stars: 366 | Forks: 163 | Open Issues: 5  

2. **[Data-Structures-Algorithms-Problem-Solving](https://github.com/singhsanket143/Data-Structures-Algorithms-Problem-Solving)**  
   Language: JavaScript  
   Stars: 253 | Forks: 60 | Open Issues: 2  

3. **[Unacademy-Notes](https://github.com/singhsanket143/Unacademy-Notes)**  
   Description: Notes of live classes.  
   Stars: 170 | Forks: 74 | Open Issues: 1  

4. **[Dev-Notes](https://github.com/singhsanket143/Dev-Notes)**  
   Stars: 97 | Forks: 35 | Open Issues: 0  

5. **[Sep-2022-Node-Batch-Notes](https://github.com/singhsanket143/Sep-2022-Node-Batch-Notes)**  
   Stars: 87 | Forks: 38 | Open Is

In [85]:
output = await agent.ainvoke({
    "messages": [
        {"role": "user", "content": "Can you list all my PRs from this repo https://github.com/berkmancenter/question_tool "},
    ],
})

In [86]:
print(output["messages"][-1].content)

Here are all your pull requests from the repository [berkmancenter/question_tool](https://github.com/berkmancenter/question_tool):

1. **[Feature remote control highlight](https://github.com/berkmancenter/question_tool/pull/134)**
   - **State:** Closed
   - **Created At:** August 13, 2019
   - **Closed At:** August 22, 2019
   - **Comments:** 4

2. **[Added moderator addition to db](https://github.com/berkmancenter/question_tool/pull/133)**
   - **State:** Open
   - **Created At:** July 25, 2019
   - **Updated At:** June 12, 2024
   - **Comments:** 0

3. **[Removed the hard limit of 4 moderators](https://github.com/berkmancenter/question_tool/pull/132)**
   - **State:** Closed
   - **Created At:** July 20, 2019
   - **Closed At:** July 24, 2019
   - **Comments:** 0
  
4. **[Feature remove icon in add moderator](https://github.com/berkmancenter/question_tool/pull/131)**
   - **State:** Closed
   - **Created At:** July 18, 2019
   - **Closed At:** August 20, 2019
   - **Comments:** 0

5

AttributeError: 'MultiServerMCPClient' object has no attribute 'list_resources'